In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import os
import yfinance as yf
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("TSLA.csv")

SAVE_DIR = 'tesla_models'
os.makedirs(SAVE_DIR, exist_ok=True)

WINDOW = 30

# ══════════════════════════════════════════════════════════════
# SHARED SCALER — fit once, reuse across all models
# ══════════════════════════════════════════════════════════════

scaler = MinMaxScaler()
price_data = df[['Adj Close']].values
scaler.fit(price_data)
joblib.dump(scaler, f'{SAVE_DIR}/shared_scaler.pkl')

data_scaled  = scaler.transform(price_data)
split_idx    = int(len(data_scaled) * 0.80)
train_data   = data_scaled[:split_idx]
test_data    = data_scaled[split_idx:]

# ══════════════════════════════════════════════════════════════
# SEQUENCE BUILDER — multi-step target
# ══════════════════════════════════════════════════════════════

def make_sequences_multistep(data, window, horizon):
    """
    window  : lookback days
    horizon : how many steps ahead to predict
    """
    X, y = [], []
    for i in range(window, len(data) - horizon + 1):
        X.append(data[i-window:i, 0])
        y.append(data[i:i+horizon, 0])   # next `horizon` steps
    return np.array(X), np.array(y)

# ══════════════════════════════════════════════════════════════
# MODEL BUILDER
# ══════════════════════════════════════════════════════════════

def build_lstm(window, horizon):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True,
                   input_shape=(window, 1)))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(horizon))             # output = horizon steps
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# ══════════════════════════════════════════════════════════════
# TRAINER
# ══════════════════════════════════════════════════════════════

def train_model(name, horizon):

    print(f"\n{'═' * 55}")
    print(f"  TRAINING {name}  |  horizon={horizon} days")
    print(f"{'═' * 55}")

    # Build sequences
    X_train, y_train = make_sequences_multistep(train_data, WINDOW, horizon)
    X_test,  y_test  = make_sequences_multistep(test_data,  WINDOW, horizon)

    # Reshape X to 3D
    X_train = X_train.reshape(X_train.shape[0], WINDOW, 1)
    X_test  = X_test.reshape(X_test.shape[0],   WINDOW, 1)

    print(f"  X_train : {X_train.shape}  y_train : {y_train.shape}")

    # Build + train
    model = build_lstm(WINDOW, horizon)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        epochs           = 100,
        batch_size       = 32,
        validation_split = 0.1,
        callbacks        = [early_stop],
        verbose          = 0             # silent training
    )

    # Evaluate — use last step of horizon for R2
    preds_scaled = model.predict(X_test, verbose=0)

    # Inverse scale all steps
    preds_all  = scaler.inverse_transform(preds_scaled)
    actual_all = scaler.inverse_transform(y_test)

    # Metrics on day-1 prediction (most reliable)
    r2   = r2_score(actual_all[:, 0],  preds_all[:, 0])
    mae  = mean_absolute_error(actual_all[:, 0], preds_all[:, 0])
    rmse = np.sqrt(mean_squared_error(actual_all[:, 0], preds_all[:, 0]))
    mape = np.mean(np.abs(
        (actual_all[:, 0] - preds_all[:, 0]) / actual_all[:, 0]
    )) * 100

    print(f"  R2   : {r2:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAPE : {mape:.2f}%")

    # Save
    model.save(f'{SAVE_DIR}/{name.lower()}.keras')
    print(f"  ✓ Saved → {SAVE_DIR}/{name.lower()}.keras")

    return model, {'R2': round(r2,4), 'MAE': round(mae,4),
                   'RMSE': round(rmse,4), 'MAPE': round(mape,2)}

# ══════════════════════════════════════════════════════════════
# TRAIN ALL 3 MODELS
# ══════════════════════════════════════════════════════════════

model_a, results_a = train_model('model_a', horizon=1)
model_b, results_b = train_model('model_b', horizon=5)
model_c, results_c = train_model('model_c', horizon=21)

# ══════════════════════════════════════════════════════════════
# SAVE METADATA
# ══════════════════════════════════════════════════════════════

metadata = {
    'window_size' : WINDOW,
    'ticker'      : 'TSLA',
    'models' : {
        'model_a' : {'horizon': 1,  'max_days': 1,  'results': results_a},
        'model_b' : {'horizon': 5,  'max_days': 5,  'results': results_b},
        'model_c' : {'horizon': 21, 'max_days': 21, 'results': results_c},
    }
}

with open(f'{SAVE_DIR}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"\n✓ Metadata saved")

# ══════════════════════════════════════════════════════════════
# PREDICTOR CLASS
# ══════════════════════════════════════════════════════════════

class TeslaPricePredictor:
    """
    Production-ready predictor. Auto-fetches live TSLA data.

    Usage
    -----
    p = TeslaPricePredictor()
    p.predict_next(days=1)    # Model A — tomorrow only
    p.predict_next(days=3)    # Model B — up to 5 days
    p.predict_next(days=15)   # Model C — up to 21 days
    """

    def __init__(self, model_dir='tesla_models'):
        self.model_dir = model_dir
        self.scaler    = joblib.load(f'{model_dir}/shared_scaler.pkl')
        self.ticker    = 'TSLA'

        with open(f'{model_dir}/metadata.json') as f:
            self.meta = json.load(f)

        self.window = self.meta['window_size']

        # Load all 3 models
        self.models = {
            'model_a': load_model(f'{model_dir}/model_a.keras'),
            'model_b': load_model(f'{model_dir}/model_b.keras'),
            'model_c': load_model(f'{model_dir}/model_c.keras'),
        }
        print(f"✓ All 3 models loaded  |  ticker={self.ticker}")

    def _fetch_prices(self):
        tsla   = yf.download(self.ticker, period='60d',
                             auto_adjust=True, progress=False)
        prices = tsla['Close'].values.flatten()
        dates  = tsla.index.tolist()
        print(f"  ✓ Fetched {len(prices)} days  |  "
              f"Latest: {dates[-1].strftime('%Y-%m-%d')}  |  "
              f"Price: ${prices[-1]:.2f}")
        return prices, dates

    def _select_model(self, days):
        """Auto select right model based on requested days."""
        if days == 1:
            return self.models['model_a'], 'Model A', 1
        elif days <= 5:
            return self.models['model_b'], 'Model B', 5
        elif days <= 21:
            return self.models['model_c'], 'Model C', 21
        else:
            raise ValueError("Maximum supported forecast: 21 days.")

    def _get_trading_dates(self, start_date, n):
        """Generate next N trading dates (skip weekends)."""
        dates, current = [], start_date
        while len(dates) < n:
            current += timedelta(days=1)
            if current.weekday() < 5:   # Mon-Fri only
                dates.append(current)
        return dates

    def predict_next(self, days=1):
        """
        Predict next N trading days.

        Parameters
        ----------
        days : int
               1       → Model A (next day only)
               2 to 5  → Model B
               6 to 21 → Model C

        Returns
        -------
        list of predicted prices
        """

        if days < 1:
            raise ValueError("days must be >= 1")

        # Fetch live data
        print(f"\n  Fetching live data...")
        prices, dates = self._fetch_prices()

        # Select model
        model, model_name, horizon = self._select_model(days)

        # Prepare input
        last_window   = prices[-self.window:]
        scaled        = self.scaler.transform(
                            last_window.reshape(-1, 1))
        X             = scaled.reshape(1, self.window, 1)

        # Predict all horizon steps
        pred_scaled   = model.predict(X, verbose=0)         # (1, horizon)
        pred_all      = self.scaler.inverse_transform(
                            pred_scaled.reshape(-1, 1)
                        ).flatten()

        # Slice only requested days
        predicted     = pred_all[:days]
        last_price    = round(float(prices[-1]), 2)
        future_dates  = self._get_trading_dates(dates[-1], days)

        # ── Print report ──────────────────────────────────
        print(f"\n{'═' * 55}")
        print(f"  TESLA FORECAST  |  {model_name}  |  {days} day(s)")
        print(f"{'═' * 55}")
        print(f"  Last close : ${last_price}  ({dates[-1].strftime('%Y-%m-%d')})")
        print(f"{'─' * 55}")
        print(f"  {'Day':<6} {'Date':<14} {'Price':>10} {'Change':>12}")
        print(f"{'─' * 55}")

        prev = last_price
        for i, (date, price) in enumerate(zip(future_dates, predicted), 1):
            price      = round(float(price), 2)
            change     = round(price - prev, 2)
            change_pct = round((change / prev) * 100, 2)
            sign       = '+' if change >= 0 else ''
            arrow      = '▲' if change >= 0 else '▼'
            print(f"  {i:<6} {date.strftime('%Y-%m-%d'):<14} "
                  f"${price:>9} {sign}{change_pct:>9}%  {arrow}")
            prev = price

        print(f"{'─' * 55}")
        total      = round(float(predicted[-1]) - last_price, 2)
        total_pct  = round((total / last_price) * 100, 2)
        sign       = '+' if total >= 0 else ''
        print(f"  {'Total':<6} {'':<14} ${round(float(predicted[-1]),2):>9} "
              f"{sign}{total_pct:>9}%  overall")
        print(f"\n  Model R2 : {self.meta['models'][model_name.lower().replace(' ','_')]['results']['R2']}")
        print(f"{'═' * 55}\n")

        return list(predicted)

    def summary(self):
        print(f"\n{'═' * 55}")
        print(f"  TESLA PREDICTOR — 3 MODEL SUMMARY")
        print(f"{'═' * 55}")
        print(f"  {'Model':<10} {'Horizon':<12} {'Max Days':<10} {'R2':>8}")
        print(f"  {'─' * 48}")
        labels = {
            'model_a': 'Model A',
            'model_b': 'Model B',
            'model_c': 'Model C'
        }
        for key, label in labels.items():
            m = self.meta['models'][key]
            print(f"  {label:<10} {m['horizon']:<12} {m['max_days']:<10} "
                  f"{m['results']['R2']:>8}")
        print(f"{'═' * 55}\n")


# ══════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════


═══════════════════════════════════════════════════════
  TRAINING model_a  |  horizon=1 days
═══════════════════════════════════════════════════════
  X_train : (1902, 30, 1)  y_train : (1902, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  R2   : 0.9674
  MAE  : 7.7066
  RMSE : 13.3443
  MAPE : 2.44%
  ✓ Saved → tesla_models/model_a.keras

═══════════════════════════════════════════════════════
  TRAINING model_b  |  horizon=5 days
═══════════════════════════════════════════════════════
  X_train : (1898, 30, 1)  y_train : (1898, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  R2   : 0.8981
  MAE  : 15.0366
  RMSE : 20.9205
  MAPE : 4.86%
  ✓ Saved → tesla_models/model_b.keras

═══════════════════════════════════════════════════════
  TRAINING model_c  |  horizon=21 days
═══════════════════════════════════════════════════════
  X_train : (1882, 30, 1)  y_train : (1882, 21)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  R2   : 0.8457
  MAE  : 15.6583
  RMSE : 19.6584
  MAPE : 5.40%
  ✓ Saved → tesla_models/model_c.keras

✓ Metadata saved


In [ ]:
p = TeslaPricePredictor('tesla_models')
p.summary()

p.predict_next(days=1)    # Model A — tomorrow
p.predict_next(days=3)    # Model B — 3 days
p.predict_next(days=15)   # Model C — 15 days

✓ All 3 models loaded  |  ticker=TSLA

═══════════════════════════════════════════════════════
  TESLA PREDICTOR — 3 MODEL SUMMARY
═══════════════════════════════════════════════════════
  Model      Horizon      Max Days         R2
  ────────────────────────────────────────────────
  Model A    1            1            0.9674
  Model B    5            5            0.8981
  Model C    21           21           0.8457
═══════════════════════════════════════════════════════


  Fetching live data...
  ✓ Fetched 60 days  |  Latest: 2026-06-09  |  Price: $396.68

═══════════════════════════════════════════════════════
  TESLA FORECAST  |  Model A  |  1 day(s)
═══════════════════════════════════════════════════════
  Last close : $396.68  (2026-06-09)
───────────────────────────────────────────────────────
  Day    Date                Price       Change
───────────────────────────────────────────────────────
  1      2026-06-10     $   397.05 +     0.09%  ▲
────────────────────────────────


═══════════════════════════════════════════════════════
  TESLA FORECAST  |  Model C  |  15 day(s)
═══════════════════════════════════════════════════════
  Last close : $396.68  (2026-06-09)
───────────────────────────────────────────────────────
  Day    Date                Price       Change
───────────────────────────────────────────────────────
  1      2026-06-10     $   383.52     -3.32%  ▼
  2      2026-06-11     $   390.24 +     1.75%  ▲
  3      2026-06-12     $   376.76     -3.45%  ▼
  4      2026-06-15     $   410.33 +     8.91%  ▲
  5      2026-06-16     $   396.42     -3.39%  ▼
  6      2026-06-17     $   398.51 +     0.53%  ▲
  7      2026-06-18     $   388.51     -2.51%  ▼
  8      2026-06-19     $   388.64 +     0.03%  ▲
  9      2026-06-22     $   382.06     -1.69%  ▼
  10     2026-06-23     $   374.28     -2.04%  ▼
  11     2026-06-24     $   376.72 +     0.65%  ▲
  12     2026-06-25     $   378.99 +      0.6%  ▲
  13     2026-06-26     $   366.75     -3.23%  ▼
  14

[np.float32(383.51688),
 np.float32(390.2358),
 np.float32(376.7562),
 np.float32(410.32785),
 np.float32(396.42184),
 np.float32(398.50626),
 np.float32(388.51056),
 np.float32(388.63736),
 np.float32(382.06155),
 np.float32(374.27588),
 np.float32(376.7242),
 np.float32(378.98563),
 np.float32(366.75333),
 np.float32(399.38284),
 np.float32(373.18286)]